In [2]:
# Data manipulation
import pandas as pd
import numpy as np

# File handling
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully")

Libraries imported successfully


In [3]:
# Project root
project_root = Path("..")

# Dataset folder
data_folder = project_root / "data" / "raw" / "2024" / "results"

# Find all CSV files
csv_files = sorted(data_folder.glob("*.csv"))

print("Dataset folder:", data_folder)
print("Number of CSV files found:", len(csv_files))

for file in csv_files:
    print(file.name)

Dataset folder: ..\data\raw\2024\results
Number of CSV files found: 12
test_result_202401.csv
test_result_202402.csv
test_result_202403.csv
test_result_202404.csv
test_result_202405.csv
test_result_202406.csv
test_result_202407.csv
test_result_202408.csv
test_result_202409.csv
test_result_202410.csv
test_result_202411.csv
test_result_202412.csv


In [4]:
# Select the first CSV file
first_file = csv_files[0]

print("Inspecting:", first_file.name)

# Read only the first 10 rows
df_sample = pd.read_csv(first_file, nrows=10)

df_sample

Inspecting: test_result_202401.csv


,test_id,vehicle_id,test_date,test_class_id,test_type,test_result,test_mileage,postcode_area,make,model,colour,fuel_type,cylinder_capacity,first_use_date,completed_date
0,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
1,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
2,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
3,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
4,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
5,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
6,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
7,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
8,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
9,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z


In [5]:
df_sample.columns.tolist()

['test_id',
 'vehicle_id',
 'test_date',
 'test_class_id',
 'test_type',
 'test_result',
 'test_mileage',
 'postcode_area',
 'make',
 'model',
 'colour',
 'fuel_type',
 'cylinder_capacity',
 'first_use_date',
 'completed_date']

In [6]:
# Check the data types of each column
df_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   test_id            10 non-null     int64  
 1   vehicle_id         10 non-null     int64  
 2   test_date          10 non-null     str    
 3   test_class_id      10 non-null     int64  
 4   test_type          10 non-null     str    
 5   test_result        10 non-null     str    
 6   test_mileage       10 non-null     float64
 7   postcode_area      10 non-null     str    
 8   make               10 non-null     str    
 9   model              10 non-null     int64  
 10  colour             10 non-null     str    
 11  fuel_type          10 non-null     str    
 12  cylinder_capacity  10 non-null     int64  
 13  first_use_date     10 non-null     str    
 14  completed_date     10 non-null     str    
dtypes: float64(1), int64(5), str(9)
memory usage: 1.3 KB


In [7]:
categorical_columns = [
    "test_class_id",
    "test_type",
    "test_result",
    "fuel_type",
    "make",
    "model",
    "colour"
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(df_sample[column].value_counts(dropna=False))


--- test_class_id ---
test_class_id
4    10
Name: count, dtype: int64

--- test_type ---
test_type
NT    10
Name: count, dtype: int64

--- test_result ---
test_result
P    10
Name: count, dtype: int64

--- fuel_type ---
fuel_type
PE    10
Name: count, dtype: int64

--- make ---
make
PORSCHE    10
Name: count, dtype: int64

--- model ---
model
911    10
Name: count, dtype: int64

--- colour ---
colour
RED    10
Name: count, dtype: int64


In [8]:
# Count important categories across the full January dataset

test_class_counts = {}
test_type_counts = {}
test_result_counts = {}

for chunk in pd.read_csv(
    first_file,
    usecols=["test_class_id", "test_type", "test_result"],
    dtype=str,
    chunksize=200_000,
    on_bad_lines="skip"
):
    for value, count in chunk["test_class_id"].value_counts().items():
        test_class_counts[value] = test_class_counts.get(value, 0) + count

    for value, count in chunk["test_type"].value_counts().items():
        test_type_counts[value] = test_type_counts.get(value, 0) + count

    for value, count in chunk["test_result"].value_counts().items():
        test_result_counts[value] = test_result_counts.get(value, 0) + count


print("TEST CLASSES")
print(test_class_counts)

print("\nTEST TYPES")
print(test_type_counts)

print("\nTEST RESULTS")
print(test_result_counts)

TEST CLASSES
{'4': 4124645, '7': 128365, '2': 33638, '1': 15095, '5': 5490, '3': 651, 'test_class_id': 11}

TEST TYPES
{'NT': 3586898, 'RT': 720959, 'test_type': 11, 'EI': 18, 'ES': 9}

TEST RESULTS
{'P': 3310945, 'F': 771976, 'PRS': 199767, 'ABR': 22367, 'ABA': 2829, 'test_result': 11}


In [9]:
# Count results for Class 4 initial MOT tests only

class4_nt_results = {}

for chunk in pd.read_csv(
    first_file,
    usecols=["test_class_id", "test_type", "test_result"],
    dtype=str,
    chunksize=200_000,
    on_bad_lines="skip"
):
    # Keep only Class 4 initial tests
    filtered = chunk[
        (chunk["test_class_id"] == "4") &
        (chunk["test_type"] == "NT")
    ]

    counts = filtered["test_result"].value_counts()

    for value, count in counts.items():
        class4_nt_results[value] = (
            class4_nt_results.get(value, 0) + count
        )

print("CLASS 4 - INITIAL TEST RESULTS")
print(class4_nt_results)

CLASS 4 - INITIAL TEST RESULTS
{'P': 2487630, 'F': 735504, 'PRS': 189097, 'ABR': 17816, 'ABA': 2669}


In [10]:
# Define the Model 1 target mapping

target_mapping = {
    "P": 0,
    "F": 1,
    "PRS": 1
}

print(target_mapping)

{'P': 0, 'F': 1, 'PRS': 1}


In [11]:
# Calculate Model 1 target balance for January

pass_count = class4_nt_results["P"]

fail_count = (
    class4_nt_results["F"] +
    class4_nt_results["PRS"]
)

total_count = pass_count + fail_count

pass_percentage = (pass_count / total_count) * 100
fail_percentage = (fail_count / total_count) * 100

print("MODEL 1 TARGET BALANCE")
print(f"Pass (0): {pass_count:,} ({pass_percentage:.2f}%)")
print(f"Fail (1): {fail_count:,} ({fail_percentage:.2f}%)")
print(f"Total usable records: {total_count:,}")

MODEL 1 TARGET BALANCE
Pass (0): 2,487,630 (72.90%)
Fail (1): 924,601 (27.10%)
Total usable records: 3,412,231


In [12]:
# Columns we want to investigate
columns_to_check = [
    "test_class_id",
    "test_type",
    "test_result",
    "test_mileage",
    "postcode_area",
    "make",
    "model",
    "colour",
    "fuel_type",
    "cylinder_capacity",
    "first_use_date"
]

missing_counts = {column: 0 for column in columns_to_check}
total_rows = 0

for chunk in pd.read_csv(
    first_file,
    usecols=columns_to_check,
    dtype=str,
    chunksize=200_000,
    on_bad_lines="skip"
):
    # Keep only records relevant to Model 1
    filtered = chunk[
        (chunk["test_class_id"] == "4") &
        (chunk["test_type"] == "NT") &
        (chunk["test_result"].isin(["P", "F", "PRS"]))
    ]

    total_rows += len(filtered)

    for column in columns_to_check:
        missing_counts[column] += filtered[column].isna().sum()


missing_summary = pd.DataFrame({
    "Missing Count": missing_counts
})

missing_summary["Missing %"] = (
    missing_summary["Missing Count"] / total_rows * 100
).round(2)

print("Total records checked:", f"{total_rows:,}")

missing_summary.sort_values(
    "Missing %",
    ascending=False
)

Total records checked: 3,412,231


,Missing Count,Missing %
cylinder_capacity,30561,0.90
test_mileage,7640,0.22
test_class_id,0,0.00
test_result,0,0.00
test_type,0,0.00
make,0,0.00
postcode_area,0,0.00
model,2,0.00
colour,0,0.00
fuel_type,0,0.00


In [13]:
# Check first_use_date missing values

first_use_missing = 0
total_rows = 0

for chunk in pd.read_csv(
    first_file,
    usecols=["test_class_id", "test_type", "test_result", "first_use_date"],
    dtype=str,
    chunksize=200_000,
    on_bad_lines="skip"
):
    filtered = chunk[
        (chunk["test_class_id"] == "4") &
        (chunk["test_type"] == "NT") &
        (chunk["test_result"].isin(["P", "F", "PRS"]))
    ]

    total_rows += len(filtered)
    first_use_missing += filtered["first_use_date"].isna().sum()

print("Total records:", f"{total_rows:,}")
print("Missing first_use_date:", f"{first_use_missing:,}")
print(
    "Missing percentage:",
    round(first_use_missing / total_rows * 100, 2),
    "%"
)

Total records: 3,412,231
Missing first_use_date: 0
Missing percentage: 0.0 %


In [14]:
# Check repeated test_id values in January dataset

test_id_counts = {}

for chunk in pd.read_csv(
    first_file,
    usecols=["test_id"],
    dtype=str,
    chunksize=200_000,
    on_bad_lines="skip"
):
    counts = chunk["test_id"].value_counts()

    for value, count in counts.items():
        test_id_counts[value] = test_id_counts.get(value, 0) + count


# Convert to DataFrame
test_id_frequency = pd.Series(test_id_counts)

print("Total unique test IDs:", len(test_id_frequency))

print("\nFrequency of test_id occurrences:")
print(test_id_frequency.value_counts().head(10))

Total unique test IDs: 2647681

Frequency of test_id occurrences:
1     1826506
2      565668
4      128716
3       51343
6       31936
8       18565
9        7929
12       7708
16       2308
18       1823
Name: count, dtype: int64


In [15]:
repeated_ids = test_id_frequency[test_id_frequency > 1].index[:5]

print("Example repeated test IDs:")
print(list(repeated_ids))

Example repeated test IDs:
['1337576829', '1620958423', '1360686225', '776023309', '830054385']


In [16]:
# Inspect rows belonging to those repeated test IDs

sample_repeated_ids = list(repeated_ids)

repeated_rows = []

for chunk in pd.read_csv(
    first_file,
    dtype=str,
    chunksize=200_000,
    on_bad_lines="skip"
):
    matched = chunk[chunk["test_id"].isin(sample_repeated_ids)]
    
    if len(matched) > 0:
        repeated_rows.append(matched)

    if len(repeated_rows) >= 1:
        break

repeated_df = pd.concat(repeated_rows)

repeated_df

,test_id,vehicle_id,test_date,test_class_id,test_type,test_result,test_mileage,postcode_area,make,model,colour,fuel_type,cylinder_capacity,first_use_date,completed_date
45192,1360686225,479251383,2024-01-03,4,NT,P,35174.0,LE,VOLKSWAGEN,TRANSPORTER,GREY,DI,1968,2019-03-01,2024-01-03T10:24:29.000Z
45193,1360686225,479251383,2024-01-03,4,NT,P,35174.0,LE,VOLKSWAGEN,TRANSPORTER,GREY,DI,1968,2019-03-01,2024-01-03T10:24:29.000Z
45194,1360686225,479251383,2024-01-03,4,NT,P,35174.0,LE,VOLKSWAGEN,TRANSPORTER,GREY,DI,1968,2019-03-01,2024-01-03T10:24:29.000Z
45195,1360686225,479251383,2024-01-03,4,NT,P,35174.0,LE,VOLKSWAGEN,TRANSPORTER,GREY,DI,1968,2019-03-01,2024-01-03T10:24:29.000Z
45196,1360686225,479251383,2024-01-03,4,NT,P,35174.0,LE,VOLKSWAGEN,TRANSPORTER,GREY,DI,1968,2019-03-01,2024-01-03T10:24:29.000Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159185,1337576829,1094694625,2024-01-19,4,NT,P,107066.0,B,ROLLS ROYCE,WRAITH,MAROON,PE,6592,2017-12-15,2024-01-19T15:02:28.000Z
159186,1337576829,1094694625,2024-01-19,4,NT,P,107066.0,B,ROLLS ROYCE,WRAITH,MAROON,PE,6592,2017-12-15,2024-01-19T15:02:28.000Z
159187,1337576829,1094694625,2024-01-19,4,NT,P,107066.0,B,ROLLS ROYCE,WRAITH,MAROON,PE,6592,2017-12-15,2024-01-19T15:02:28.000Z
159188,1337576829,1094694625,2024-01-19,4,NT,P,107066.0,B,ROLLS ROYCE,WRAITH,MAROON,PE,6592,2017-12-15,2024-01-19T15:02:28.000Z


In [17]:
# Check exact duplicate rows in the January file sample

duplicate_count = 0
total_count = 0

for chunk in pd.read_csv(
    first_file,
    dtype=str,
    chunksize=200_000,
    on_bad_lines="skip"
):
    total_count += len(chunk)
    duplicate_count += chunk.duplicated().sum()

print("Total rows scanned:", f"{total_count:,}")
print("Exact duplicate rows:", f"{duplicate_count:,}")
print(
    "Duplicate percentage:",
    round((duplicate_count / total_count) * 100, 2),
    "%"
)

Total rows scanned: 4,307,894
Exact duplicate rows: 1,614,728
Duplicate percentage: 37.48 %


In [18]:
def clean_mot_data(df):
    """
    Clean raw DVSA MOT results data.
    """

    # Remove repeated header rows
    df = df[df["test_id"] != "test_id"]

    # Remove exact duplicate rows
    df = df.drop_duplicates()

    # Keep only Class 4 initial tests
    df = df[
        (df["test_class_id"] == "4") &
        (df["test_type"] == "NT")
    ]

    # Keep valid outcomes only
    df = df[
        df["test_result"].isin(["P", "F", "PRS"])
    ]

    return df

In [19]:
id="step14"
# Test cleaning function on January dataset

cleaned_chunks = []

for chunk in pd.read_csv(
    first_file,
    dtype=str,
    chunksize=200_000,
    on_bad_lines="skip"
):
    cleaned_chunk = clean_mot_data(chunk)

    cleaned_chunks.append(cleaned_chunk)

# Combine cleaned chunks
jan_cleaned = pd.concat(
    cleaned_chunks,
    ignore_index=True
)

print("Original file:", first_file.name)
print("Cleaned rows:", f"{len(jan_cleaned):,}")

jan_cleaned.head()

Original file: test_result_202401.csv
Cleaned rows: 2,115,823


,test_id,vehicle_id,test_date,test_class_id,test_type,test_result,test_mileage,postcode_area,make,model,colour,fuel_type,cylinder_capacity,first_use_date,completed_date
0,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
1,300046445,844431602,2024-01-23,4,NT,P,133969.0,G,BMW,3 SERIES,GREEN,PE,1796,1998-01-06,2024-01-23T10:54:36.000Z
2,300046445,844431602,2024-01-23,4,NT,P,133969.0,G,BMW,3 SERIES,GREEN,PE,1796,1998-12-31,2024-01-23T10:54:36.000Z
3,920478433,1168307186,2024-01-17,4,NT,PRS,82557.0,CM,FIAT,500,WHITE,DI,1248,2010-03-10,2024-01-17T16:09:06.000Z
4,1399602761,1343858296,2024-01-26,4,NT,P,24967.0,SG,ALFA ROMEO,SPIDER,RED,PE,2198,2008-01-26,2024-01-26T09:48:52.000Z


In [20]:
# Create the target variable
# P   = 0 -> Initial pass
# F   = 1 -> Initial fail
# PRS = 1 -> Initially failed, then rectified

jan_cleaned["initial_fail"] = jan_cleaned["test_result"].map({
    "P": 0,
    "F": 1,
    "PRS": 1
})

# Convert dates to datetime
jan_cleaned["test_date"] = pd.to_datetime(
    jan_cleaned["test_date"],
    errors="coerce"
)

jan_cleaned["first_use_date"] = pd.to_datetime(
    jan_cleaned["first_use_date"],
    errors="coerce"
)

# Calculate vehicle age in years
jan_cleaned["vehicle_age_years"] = (
    jan_cleaned["test_date"] - jan_cleaned["first_use_date"]
).dt.days / 365.25

# Display the result
jan_cleaned[
    [
        "test_date",
        "first_use_date",
        "vehicle_age_years",
        "test_result",
        "initial_fail"
    ]
].head(10)

,test_date,first_use_date,vehicle_age_years,test_result,initial_fail
0,2024-01-02,2010-09-01,13.336071,P,0
1,2024-01-23,1998-01-06,26.045175,P,0
2,2024-01-23,1998-12-31,25.062286,P,0
3,2024-01-17,2010-03-10,13.856263,PRS,1
4,2024-01-26,2008-01-26,16.000000,P,0
5,2024-01-09,2008-12-17,15.060917,F,1
6,2024-01-26,2002-09-11,21.374401,F,1
7,2024-01-26,2002-09-11,21.374401,F,1
8,2024-01-31,2011-03-07,12.903491,P,0
9,2024-01-10,2010-01-20,13.971253,P,0


In [21]:
# Check vehicle age quality

print("Missing vehicle ages:")
print(jan_cleaned["vehicle_age_years"].isna().sum())

print("\nNegative vehicle ages:")
print((jan_cleaned["vehicle_age_years"] < 0).sum())

print("\nZero vehicle ages:")
print((jan_cleaned["vehicle_age_years"] == 0).sum())

print("\nVehicle age summary:")
print(jan_cleaned["vehicle_age_years"].describe())

Missing vehicle ages:
0

Negative vehicle ages:
0

Zero vehicle ages:
44

Vehicle age summary:
count    2.115823e+06
mean     1.055732e+01
std      5.511983e+00
min      0.000000e+00
25%      6.477755e+00
50%      9.708419e+00
75%      1.385900e+01
max      1.013837e+03
Name: vehicle_age_years, dtype: float64


In [22]:
# Inspect extremely large vehicle ages

suspicious_age = jan_cleaned[
    jan_cleaned["vehicle_age_years"] > 100
][
    [
        "test_id",
        "vehicle_id",
        "test_date",
        "first_use_date",
        "vehicle_age_years",
        "make",
        "model",
        "test_result"
    ]
]

print("Number of records with age > 100 years:",
      len(suspicious_age))

suspicious_age.head(20)

Number of records with age > 100 years: 5


,test_id,vehicle_id,test_date,first_use_date,vehicle_age_years,make,model,test_result
263232,1314504987,641918509,2024-01-03,1918-02-05,105.908282,FIAT,DOBLO,F
299475,1380328051,1358976330,2024-01-16,1923-12-16,100.084873,BENTLEY,UNCLASSIFIED,P
530097,939540075,342909472,2024-01-25,1920-06-15,103.611225,PEUGEOT,206,F
1209655,927224391,1345159291,2024-01-08,1010-03-01,1013.837098,HONDA,INSIGHT,PRS
2046491,452628783,515121789,2024-01-18,1906-01-10,118.020534,SUBARU,IMPREZA,P


In [23]:
# Remove invalid vehicle ages

before_count = len(jan_cleaned)

jan_cleaned = jan_cleaned[
    (jan_cleaned["vehicle_age_years"] > 0) &
    (jan_cleaned["vehicle_age_years"] <= 100)
].copy()

after_count = len(jan_cleaned)

print("Rows before age filtering:", f"{before_count:,}")
print("Rows after age filtering:", f"{after_count:,}")
print("Rows removed:", f"{before_count - after_count:,}")

Rows before age filtering: 2,115,823
Rows after age filtering: 2,115,774
Rows removed: 49


In [24]:
# Convert numerical columns to numeric values

numeric_columns = [
    "test_mileage",
    "cylinder_capacity"
]

for column in numeric_columns:
    jan_cleaned[column] = pd.to_numeric(
        jan_cleaned[column],
        errors="coerce"
    )

# Check data types
print("Data types:")
print(jan_cleaned[numeric_columns].dtypes)

# Check missing values
print("\nMissing values:")
print(jan_cleaned[numeric_columns].isna().sum())

# Check zero values
print("\nZero values:")
print((jan_cleaned[numeric_columns] == 0).sum())

# Check negative values
print("\nNegative values:")
print((jan_cleaned[numeric_columns] < 0).sum())

# Summary statistics
print("\nSummary statistics:")
print(jan_cleaned[numeric_columns].describe())

Data types:
test_mileage         float64
cylinder_capacity    float64
dtype: object

Missing values:
test_mileage          4772
cylinder_capacity    18328
dtype: int64

Zero values:
test_mileage           0
cylinder_capacity    600
dtype: int64

Negative values:
test_mileage         0
cylinder_capacity    0
dtype: int64

Summary statistics:
       test_mileage  cylinder_capacity
count  2.111002e+06       2.097446e+06
mean   7.792894e+04       1.725473e+03
std    4.676832e+04       5.943709e+02
min    1.000000e+00       0.000000e+00
25%    4.259800e+04       1.364000e+03
50%    7.022100e+04       1.598000e+03
75%    1.044160e+05       1.995000e+03
max    9.999990e+05       8.200000e+04


In [25]:
# Inspect unusually high mileage and engine capacity

print("Records with mileage >= 500,000:")
high_mileage = jan_cleaned[
    jan_cleaned["test_mileage"] >= 500_000
]

print(high_mileage[
    [
        "test_id",
        "vehicle_id",
        "test_mileage",
        "make",
        "model",
        "fuel_type",
        "cylinder_capacity",
        "test_result"
    ]
].head(20))


print("\nRecords with cylinder capacity >= 10,000 cc:")
high_engine = jan_cleaned[
    jan_cleaned["cylinder_capacity"] >= 10_000
]

print(high_engine[
    [
        "test_id",
        "vehicle_id",
        "test_mileage",
        "make",
        "model",
        "fuel_type",
        "cylinder_capacity",
        "test_result"
    ]
].head(20))

Records with mileage >= 500,000:
           test_id  vehicle_id  test_mileage              make  \
32686    180735159  1490230218      621358.0              AUDI   
39783    951111399   571155634      578280.0            TOYOTA   
68909    404260265     4257400      503229.0              FORD   
72860   1891638821  1010544254      736420.0           RENAULT   
89442   1779690671   541743711      510501.0           PEUGEOT   
117173   967143505  1308820062      537403.0  LONDON TAXIS INT   
128133  1801754983   975101821      820490.0     MERCEDES-BENZ   
130939   411107301   102070864      583235.0     MERCEDES-BENZ   
135081    28342989   532997429      852007.0        VOLKSWAGEN   
138642  1781549711   830712703      867321.0               KIA   
140723   481093515   119070514      517908.0     MERCEDES-BENZ   
162289   732265543   420059872      527703.0  LONDON TAXIS INT   
166953  1304483061   219962977      520458.0     MERCEDES-BENZ   
204395   775463493    18814755      564177.

In [26]:
# Inspect extremely high engine capacities

extreme_engine = jan_cleaned[
    jan_cleaned["cylinder_capacity"] >= 10_000
].sort_values(
    "cylinder_capacity",
    ascending=False
)

print(
    "Number of records with cylinder capacity >= 10,000 cc:",
    len(extreme_engine)
)

extreme_engine[
    [
        "test_id",
        "vehicle_id",
        "make",
        "model",
        "fuel_type",
        "cylinder_capacity",
        "test_mileage",
        "vehicle_age_years",
        "test_result"
    ]
].head(30)

Number of records with cylinder capacity >= 10,000 cc: 34


,test_id,vehicle_id,make,model,fuel_type,cylinder_capacity,test_mileage,vehicle_age_years,test_result
1027472,880320259,1269368471,CADILLAC,ELDARADO,PE,82000.0,85547.0,4.818617,P
891568,158936549,1347489287,DODGE,RAM,PE,50007.0,143330.0,8.887064,P
1620188,559273783,771796783,JAGUAR,XJ,PE,50003.0,37780.0,43.036277,P
525744,1054696031,8702139,FORD,RANGER,DI,32000.0,62743.0,7.561944,P
57639,208655715,48335865,LAND ROVER,DEFENDER,DI,25000.0,104121.0,27.920602,P
30833,1619227887,459768327,TOYOTA,ESTIMA,PE,23600.0,60515.0,7.947981,P
33532,1850764363,1188738225,TOYOTA,ESTIMA,PE,23600.0,76335.0,7.455168,P
1632935,1128147059,742955977,TOYOTA,ESTIMA,PE,23500.0,26290.0,7.868583,P
1825307,1031728135,794685181,FIAT,HYMER,DI,22887.0,4140.0,3.890486,P
238777,57449569,1112967421,JAGUAR,X-TYPE,PE,21000.0,54143.0,14.915811,P


In [27]:
# Inspect extremely high engine capacities

extreme_engine = jan_cleaned[
    jan_cleaned["cylinder_capacity"] >= 10_000
].sort_values(
    "cylinder_capacity",
    ascending=False
)

print(
    "Number of records with cylinder capacity >= 10,000 cc:",
    len(extreme_engine)
)

extreme_engine[
    [
        "test_id",
        "vehicle_id",
        "make",
        "model",
        "fuel_type",
        "cylinder_capacity",
        "test_mileage",
        "vehicle_age_years",
        "test_result"
    ]
].head(30)

Number of records with cylinder capacity >= 10,000 cc: 34


,test_id,vehicle_id,make,model,fuel_type,cylinder_capacity,test_mileage,vehicle_age_years,test_result
1027472,880320259,1269368471,CADILLAC,ELDARADO,PE,82000.0,85547.0,4.818617,P
891568,158936549,1347489287,DODGE,RAM,PE,50007.0,143330.0,8.887064,P
1620188,559273783,771796783,JAGUAR,XJ,PE,50003.0,37780.0,43.036277,P
525744,1054696031,8702139,FORD,RANGER,DI,32000.0,62743.0,7.561944,P
57639,208655715,48335865,LAND ROVER,DEFENDER,DI,25000.0,104121.0,27.920602,P
30833,1619227887,459768327,TOYOTA,ESTIMA,PE,23600.0,60515.0,7.947981,P
33532,1850764363,1188738225,TOYOTA,ESTIMA,PE,23600.0,76335.0,7.455168,P
1632935,1128147059,742955977,TOYOTA,ESTIMA,PE,23500.0,26290.0,7.868583,P
1825307,1031728135,794685181,FIAT,HYMER,DI,22887.0,4140.0,3.890486,P
238777,57449569,1112967421,JAGUAR,X-TYPE,PE,21000.0,54143.0,14.915811,P


In [28]:
# Treat implausible engine capacities as missing

invalid_capacity = (
    (jan_cleaned["cylinder_capacity"] == 0) |
    (jan_cleaned["cylinder_capacity"] >= 10_000)
)

print(
    "Invalid cylinder capacity records:",
    invalid_capacity.sum()
)

jan_cleaned.loc[
    invalid_capacity,
    "cylinder_capacity"
] = np.nan

print(
    "Missing cylinder capacity after cleaning:",
    jan_cleaned["cylinder_capacity"].isna().sum()
)

Invalid cylinder capacity records: 634
Missing cylinder capacity after cleaning: 18962


In [29]:
# Investigate the maximum mileage value

max_mileage_count = (
    jan_cleaned["test_mileage"] == 999_999
).sum()

print(
    "Records with mileage = 999,999:",
    f"{max_mileage_count:,}"
)

max_mileage_records = jan_cleaned[
    jan_cleaned["test_mileage"] == 999_999
]

max_mileage_records[
    [
        "test_id",
        "vehicle_id",
        "make",
        "model",
        "fuel_type",
        "cylinder_capacity",
        "vehicle_age_years",
        "test_result"
    ]
].head(20)

Records with mileage = 999,999: 8


,test_id,vehicle_id,make,model,fuel_type,cylinder_capacity,vehicle_age_years,test_result
1269846,74974809,444125708,AUDI,TT,PE,1781.0,21.738535,F
1371344,1151515659,113771524,PEUGEOT,406,DI,1997.0,23.019849,P
1423594,920292231,1041453046,AUDI,TT,PE,1781.0,17.834360,PRS
1435215,300235291,882855828,BMW,X5,DI,2993.0,19.802875,F
1478291,1249697161,1189511326,ROVER,75,DI,1951.0,21.305955,F
1602851,410188189,370886848,BMW,X5,DI,2993.0,18.398357,P
2000353,1397570627,173210574,ROVER,75,DI,1951.0,19.835729,F
2072556,661736363,813204314,REVA,UNCLASSIFIED,EL,NaN,16.498289,P


In [30]:
# Examine mileage distribution

print("Mileage percentiles:")

print(
    jan_cleaned["test_mileage"].quantile(
        [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999]
    )
)

Mileage percentiles:
0.010      8978.00
0.050     18436.00
0.250     42598.00
0.500     70221.00
0.750    104416.00
0.950    163391.00
0.990    216897.97
0.999    311337.00
Name: test_mileage, dtype: float64


In [31]:
# Check cylinder capacity after cleaning

print("Missing cylinder capacity:")
print(jan_cleaned["cylinder_capacity"].isna().sum())

print("\nCylinder capacity percentiles:")

print(
    jan_cleaned["cylinder_capacity"].quantile(
        [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999]
    )
)

print("\nSummary statistics:")
print(
    jan_cleaned["cylinder_capacity"].describe()
)

Missing cylinder capacity:
18962

Cylinder capacity percentiles:
0.010     989.0
0.050     998.0
0.250    1368.0
0.500    1598.0
0.750    1995.0
0.950    2979.0
0.990    3800.0
0.999    5998.0
Name: cylinder_capacity, dtype: float64

Summary statistics:
count    2.096812e+06
mean     1.725655e+03
std      5.860017e+02
min      1.000000e+00
25%      1.368000e+03
50%      1.598000e+03
75%      1.995000e+03
max      9.999000e+03
Name: cylinder_capacity, dtype: float64


In [32]:
# Examine frequency of make and model categories

print("TOP 20 MAKES")
print(
    jan_cleaned["make"]
    .value_counts()
    .head(20)
)

print("\nTOP 20 MODELS")
print(
    jan_cleaned["model"]
    .value_counts()
    .head(20)
)

TOP 20 MAKES
make
FORD             277169
VOLKSWAGEN       218979
VAUXHALL         193536
NISSAN           121508
MERCEDES-BENZ    108606
TOYOTA           108315
BMW              106996
AUDI             103988
PEUGEOT           96791
CITROEN           70563
LAND ROVER        67878
RENAULT           63509
KIA               62270
SKODA             57376
MINI              51954
HYUNDAI           48544
HONDA             48121
FIAT              35426
VOLVO             35383
SEAT              35078
Name: count, dtype: int64

TOP 20 MODELS
model
FIESTA         76149
GOLF           63235
FOCUS          60680
POLO           52100
ASTRA          49984
QASHQAI        45121
CORSA          42890
YARIS          33781
C              26244
CIVIC          21668
MICRA          20189
CLIO           19806
A4             19525
MINI           19147
TRANSPORTER    19063
E              18454
FABIA          18417
A3             18273
JUKE           17206
3 SERIES       16999
Name: count, dtype: int64


In [33]:
# Analyze rare categories

make_counts = jan_cleaned["make"].value_counts()
model_counts = jan_cleaned["model"].value_counts()

print("MAKES")
print("Total unique makes:", len(make_counts))
print("Makes appearing once:", (make_counts == 1).sum())
print("Makes appearing <= 5 times:", (make_counts <= 5).sum())
print("Makes appearing <= 10 times:", (make_counts <= 10).sum())

print("\nMODELS")
print("Total unique models:", len(model_counts))
print("Models appearing once:", (model_counts == 1).sum())
print("Models appearing <= 5 times:", (model_counts <= 5).sum())
print("Models appearing <= 10 times:", (model_counts <= 10).sum())

MAKES
Total unique makes: 813
Makes appearing once: 482
Makes appearing <= 5 times: 652
Makes appearing <= 10 times: 684

MODELS
Total unique models: 7373
Models appearing once: 2420
Models appearing <= 5 times: 4211
Models appearing <= 10 times: 4885


In [34]:
# Examine fuel type distribution

fuel_counts = (
    jan_cleaned["fuel_type"]
    .value_counts(dropna=False)
)

print("Fuel type distribution:")
print(fuel_counts)

Fuel type distribution:
fuel_type
PE                         1067136
DI                          951579
HY                           53473
Hybrid Electric (Clean)      20373
EL                           13954
Electric                      4568
ED                            3114
OT                             855
LP                             559
GB                             116
FC                              19
GA                              11
GD                              11
CN                               5
ST                               1
Name: count, dtype: int64


In [35]:
# Examine postcode area distribution

postcode_counts = (
    jan_cleaned["postcode_area"]
    .value_counts(dropna=False)
)

print("Number of postcode areas:",
      jan_cleaned["postcode_area"].nunique(dropna=True))

print("\nTop 30 postcode areas:")
print(postcode_counts.head(30))

print("\nMissing postcode areas:",
      jan_cleaned["postcode_area"].isna().sum())

Number of postcode areas: 119

Top 30 postcode areas:
postcode_area
B     65525
S     47860
NG    39584
PE    38921
LE    38025
BS    36961
CF    35194
NE    34390
RG    32448
G     31848
CV    31499
M     31181
PO    30429
GU    29241
NR    29086
TN    28733
DE    28574
NN    28468
DN    28230
SA    26933
GL    26738
BN    26077
SO    25761
IP    25343
EH    25189
CM    24973
OX    24652
EX    24624
CH    23979
ST    23843
Name: count, dtype: int64

Missing postcode areas: 0


In [36]:
# Examine colour distribution

colour_counts = (
    jan_cleaned["colour"]
    .value_counts(dropna=False)
)

print("Number of colours:",
      jan_cleaned["colour"].nunique(dropna=True))

print("\nColour distribution:")
print(colour_counts)

print("\nMissing colours:",
      jan_cleaned["colour"].isna().sum())

Number of colours: 20

Colour distribution:
colour
BLACK           418248
WHITE           383088
GREY            333920
BLUE            331109
SILVER          328892
RED             209003
GREEN            35299
ORANGE           15016
BROWN            12479
BEIGE            12190
YELLOW            9831
PURPLE            6647
BRONZE            6423
GOLD              5567
TURQUOISE         2371
CREAM             2346
MAROON            1663
PINK              1263
MULTI-COLOUR       414
NOT STATED           5
Name: count, dtype: int64

Missing colours: 0


In [37]:
# Final data-quality summary

print("=" * 60)
print("FINAL DATA QUALITY SUMMARY")
print("=" * 60)

print(f"\nFinal number of records: {len(jan_cleaned):,}")
print(f"Number of columns: {jan_cleaned.shape[1]}")

print("\nMissing values:")
print(
    jan_cleaned[
        [
            "test_mileage",
            "cylinder_capacity",
            "make",
            "model",
            "fuel_type",
            "postcode_area",
            "colour",
            "vehicle_age_years"
        ]
    ].isna().sum()
)

print("\nTarget distribution:")
print(jan_cleaned["initial_fail"].value_counts())

print("\nTarget percentages:")
print(
    jan_cleaned["initial_fail"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nFinal numerical summary:")
print(
    jan_cleaned[
        [
            "test_mileage",
            "cylinder_capacity",
            "vehicle_age_years"
        ]
    ].describe()
)

FINAL DATA QUALITY SUMMARY

Final number of records: 2,115,774
Number of columns: 17

Missing values:
test_mileage          4772
cylinder_capacity    18962
make                     0
model                    1
fuel_type                0
postcode_area            0
colour                   0
vehicle_age_years        0
dtype: int64

Target distribution:
initial_fail
0    1483778
1     631996
Name: count, dtype: int64

Target percentages:
initial_fail
0    70.13
1    29.87
Name: proportion, dtype: float64

Final numerical summary:
       test_mileage  cylinder_capacity  vehicle_age_years
count  2.111002e+06       2.096812e+06       2.115774e+06
mean   7.792894e+04       1.725655e+03       1.055688e+01
std    4.676832e+04       5.860017e+02       5.466896e+00
min    1.000000e+00       1.000000e+00       2.737851e-03
25%    4.259800e+04       1.368000e+03       6.477755e+00
50%    7.022100e+04       1.598000e+03       9.708419e+00
75%    1.044160e+05       1.995000e+03       1.385900e+01
max